# Lagrangian Dynamics for Robots
## From Variational Principles to Computed Torque Control

This notebook provides a complete, from-scratch treatment of **Lagrangian dynamics** as applied to robotic manipulators. We develop the theory systematically and implement simulations for single pendulums, double pendulums, and actuated robot arms.

**What you'll learn:**
1. Why the Lagrangian formulation is preferred over Newtonian mechanics for multi-body systems
2. How to derive equations of motion using the Euler-Lagrange equation
3. The standard manipulator equation: $M(q)\ddot{q} + C(q,\dot{q})\dot{q} + g(q) = \tau$
4. Energy conservation and its role in verifying simulations
5. Computed torque control: canceling nonlinear dynamics for precise trajectory tracking

**Prerequisites:** Multivariable calculus, linear algebra, basic differential equations, calculus of variations (see [Calculus of Variations notebook](../../maths/calculus-of-variations/)).

**References:**
- Goldstein, Poole & Safko, *Classical Mechanics*, 3rd ed., Addison-Wesley, 2002.
- Murray, Li & Sastry, *A Mathematical Introduction to Robotic Manipulation*, CRC Press, 1994.
- Spong, Hutchinson & Vidyasagar, *Robot Modeling and Control*, Wiley, 2006.

---
## 1. Why Lagrangian Mechanics?

Newton's second law, $\mathbf{F} = m\mathbf{a}$, works well for a single particle. But for a robot arm with $n$ links connected by joints, the Newtonian approach requires:

- Tracking the full 3D position and orientation of every link
- Computing **constraint forces** at every joint (forces that do no work but enforce the joint geometry)
- Solving a coupled system where these unknown constraint forces appear in every equation

The **Lagrangian formulation** avoids all of this by:

| Feature | Newtonian | Lagrangian |
|---------|-----------|------------|
| **Coordinates** | Cartesian $(x, y, z)$ for each body | Generalized (joint angles $q_1, \ldots, q_n$) |
| **Constraint forces** | Must compute explicitly | Eliminated automatically |
| **Scalars vs. vectors** | Vector forces and accelerations | Scalar energies $T$ and $V$ |
| **Systematic** | Ad hoc free-body diagrams | Algorithmic: write $L$, apply EL equation |
| **Number of equations** | $3 \times$ (number of bodies) | $n$ (number of DOFs) |

For an $n$-DOF robot, we only need $n$ **generalized coordinates** $q = (q_1, \ldots, q_n)$ — typically the joint angles. The Lagrangian approach gives us exactly $n$ equations of motion, one for each DOF, with no constraint forces to worry about.

The key idea comes from the **calculus of variations** (see our [dedicated notebook](../../maths/calculus-of-variations/)): among all possible trajectories $q(t)$ connecting two configurations, the physical trajectory is the one that makes the **action** $S = \int L \, dt$ stationary.

---
## 2. The Lagrangian

The **Lagrangian** is defined as the difference between kinetic and potential energy:

$$\boxed{L(q, \dot{q}) = T(q, \dot{q}) - V(q)}$$

### Example: Simple Pendulum

Consider a point mass $m$ on a rigid, massless rod of length $l$, swinging in a plane under gravity $g$. The single generalized coordinate is the angle $\theta$ from the downward vertical.

**Position** of the mass:
$$x = l \sin\theta, \qquad y = -l \cos\theta$$

**Velocity** (by differentiation):
$$\dot{x} = l\dot{\theta}\cos\theta, \qquad \dot{y} = l\dot{\theta}\sin\theta$$

**Kinetic energy:**
$$T = \frac{1}{2}m(\dot{x}^2 + \dot{y}^2) = \frac{1}{2}m l^2 \dot{\theta}^2$$

**Potential energy** (with $y = 0$ as reference):
$$V = mgy = -mgl\cos\theta$$

**Lagrangian:**
$$\boxed{L = \frac{1}{2}ml^2\dot{\theta}^2 + mgl\cos\theta}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

np.random.seed(42)

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

# Gravitational acceleration
GRAVITY = 9.81

# Single pendulum parameters
SP_MASS = 1.0        # kg
SP_LENGTH = 1.0      # m

# Double pendulum parameters
DP_M1 = 1.0          # kg
DP_M2 = 1.0          # kg
DP_L1 = 1.0          # m
DP_L2 = 1.0          # m

# 2R planar robot arm parameters
ARM_M1 = 1.0         # kg
ARM_M2 = 0.8         # kg
ARM_L1 = 1.0         # m
ARM_L2 = 0.8         # m
ARM_LC1 = 0.5        # m, center of mass distance from joint 1
ARM_LC2 = 0.4        # m, center of mass distance from joint 2
ARM_I1 = ARM_M1 * ARM_L1**2 / 12.0   # kg*m^2, moment of inertia link 1
ARM_I2 = ARM_M2 * ARM_L2**2 / 12.0   # kg*m^2, moment of inertia link 2

# Simulation parameters
DT = 0.001           # s, integration time step
T_SIM = 10.0         # s, simulation duration
ENERGY_TOL = 1e-6    # energy conservation tolerance

In [ ]:
def pendulum_lagrangian(theta, theta_dot, m, l, g):
    """Compute kinetic energy, potential energy, and Lagrangian for a simple pendulum.

    Args:
        theta: Angle from downward vertical. Scalar or Shape: (N,).
        theta_dot: Angular velocity. Scalar or Shape: (N,).
        m: Mass. Scalar.
        l: Length. Scalar.
        g: Gravitational acceleration. Scalar.

    Returns:
        T: Kinetic energy. Same shape as theta.
        V: Potential energy. Same shape as theta.
        L: Lagrangian (T - V). Same shape as theta.
    """
    T = 0.5 * m * l**2 * theta_dot**2
    V = -m * g * l * np.cos(theta)
    L = T - V
    return T, V, L


# ---- Verify Lagrangian computation ----
theta_test = np.pi / 3
theta_dot_test = 1.5
T_test, V_test, L_test = pendulum_lagrangian(theta_test, theta_dot_test, SP_MASS, SP_LENGTH, GRAVITY)

T_expected = 0.5 * SP_MASS * SP_LENGTH**2 * theta_dot_test**2
V_expected = -SP_MASS * GRAVITY * SP_LENGTH * np.cos(theta_test)

status_T = "PASS" if abs(T_test - T_expected) < 1e-12 else "FAIL"
status_V = "PASS" if abs(V_test - V_expected) < 1e-12 else "FAIL"
print(f"Kinetic energy:   T = {T_test:.4f} J [{status_T}]")
print(f"Potential energy: V = {V_test:.4f} J [{status_V}]")
print(f"Lagrangian:       L = {L_test:.4f} J")

---
## 3. The Euler-Lagrange Equation

From the calculus of variations, the trajectory that makes the action $S = \int_0^T L(q, \dot{q}) \, dt$ stationary satisfies the **Euler-Lagrange equation** for each generalized coordinate $q_i$:

$$\boxed{\frac{d}{dt}\frac{\partial L}{\partial \dot{q}_i} - \frac{\partial L}{\partial q_i} = \tau_i}$$

where $\tau_i$ is the generalized force (torque) applied at joint $i$. For an unforced system, $\tau_i = 0$.

### Application to the Simple Pendulum

With $L = \frac{1}{2}ml^2\dot{\theta}^2 + mgl\cos\theta$:

$$\frac{\partial L}{\partial \dot{\theta}} = ml^2\dot{\theta}, \qquad \frac{d}{dt}\frac{\partial L}{\partial \dot{\theta}} = ml^2\ddot{\theta}$$

$$\frac{\partial L}{\partial \theta} = -mgl\sin\theta$$

The Euler-Lagrange equation gives:

$$ml^2\ddot{\theta} + mgl\sin\theta = \tau$$

For the unforced case ($\tau = 0$), dividing by $ml^2$:

$$\boxed{\ddot{\theta} = -\frac{g}{l}\sin\theta}$$

This is exactly Newton's second law for rotational motion ($I\alpha = -mgl\sin\theta$ with $I = ml^2$) — but we arrived at it systematically from scalar energy functions rather than vector force analysis.

**Small-angle approximation:** For $|\theta| \ll 1$, $\sin\theta \approx \theta$, giving $\ddot{\theta} \approx -(g/l)\theta$ — simple harmonic motion with period $T = 2\pi\sqrt{l/g}$.

---
## 4. Single Pendulum Simulation

In [ ]:
def rk4_step(f, t, y, dt):
    """Perform one step of 4th-order Runge-Kutta integration.

    Args:
        f: Dynamics function f(t, y) -> dy/dt. Returns Shape: (n,).
        t: Current time. Scalar.
        y: Current state. Shape: (n,).
        dt: Time step. Scalar.

    Returns:
        y_next: State at t + dt. Shape: (n,).
    """
    k1 = f(t, y)
    k2 = f(t + 0.5 * dt, y + 0.5 * dt * k1)
    k3 = f(t + 0.5 * dt, y + 0.5 * dt * k2)
    k4 = f(t + dt, y + dt * k3)
    return y + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def rk4_integrate(f, y0, t_span, dt):
    """Integrate an ODE using 4th-order Runge-Kutta.

    Args:
        f: Dynamics function f(t, y) -> dy/dt. Returns Shape: (n,).
        y0: Initial state. Shape: (n,).
        t_span: Tuple (t_start, t_end). Time interval.
        dt: Time step. Scalar.

    Returns:
        t_arr: Time array. Shape: (N_steps+1,).
        y_arr: State trajectory. Shape: (N_steps+1, n).
    """
    t_start, t_end = t_span
    N_steps = int(np.ceil((t_end - t_start) / dt))
    t_arr = np.linspace(t_start, t_end, N_steps + 1)
    y_arr = np.zeros((N_steps + 1, len(y0)))
    y_arr[0] = y0

    y = np.array(y0, dtype=float)
    for i in range(N_steps):
        y = rk4_step(f, t_arr[i], y, t_arr[i + 1] - t_arr[i])
        y_arr[i + 1] = y

    return t_arr, y_arr


print("RK4 integrator defined.")

In [ ]:
def pendulum_dynamics(t, state, m, l, g, tau=0.0):
    """Equations of motion for a simple pendulum.

    State: [theta, theta_dot]
    Equation: theta_ddot = -(g/l)*sin(theta) + tau/(m*l^2)

    Args:
        t: Current time. Scalar.
        state: [theta, theta_dot]. Shape: (2,).
        m: Mass. Scalar.
        l: Length. Scalar.
        g: Gravitational acceleration. Scalar.
        tau: Applied torque. Scalar.

    Returns:
        dstate: [theta_dot, theta_ddot]. Shape: (2,).
    """
    theta, theta_dot = state
    theta_ddot = -(g / l) * np.sin(theta) + tau / (m * l**2)
    return np.array([theta_dot, theta_ddot])


# ---- Simulate single pendulum ----
theta_0 = np.pi / 3   # 60 degrees
theta_dot_0 = 0.0

sp_dynamics = lambda t, state: pendulum_dynamics(t, state, SP_MASS, SP_LENGTH, GRAVITY)
t_sp, y_sp = rk4_integrate(sp_dynamics, [theta_0, theta_dot_0], (0, T_SIM), DT)

theta_sp = y_sp[:, 0]
theta_dot_sp = y_sp[:, 1]

# ---- Energy conservation check ----
T_sp, V_sp, L_sp = pendulum_lagrangian(theta_sp, theta_dot_sp, SP_MASS, SP_LENGTH, GRAVITY)
H_sp = T_sp + V_sp  # Hamiltonian (total energy)
H0_sp = H_sp[0]
energy_error_sp = np.max(np.abs((H_sp - H0_sp) / H0_sp))
status_energy = "PASS" if energy_error_sp < ENERGY_TOL else "FAIL"
print(f"Energy conservation: max |DeltaH/H0| = {energy_error_sp:.2e} [{status_energy}]")
print(f"Initial energy: H0 = {H0_sp:.6f} J")
print(f"Simulation: {len(t_sp)} steps, dt = {DT}")

In [ ]:
# ---- 3-Panel: Single pendulum results ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: angle vs time
ax = axes[0]
ax.plot(t_sp, np.degrees(theta_sp), color='steelblue', linewidth=2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Single Pendulum: Angle vs Time')
ax.axhline(0, color='black', linewidth=0.5)

# Panel 2: phase portrait
ax = axes[1]
ax.plot(np.degrees(theta_sp), np.degrees(theta_dot_sp), color='coral', linewidth=1.5)
ax.plot(np.degrees(theta_sp[0]), np.degrees(theta_dot_sp[0]), 'ko', markersize=8, label='Start')
ax.set_xlabel(r'$\theta$ (degrees)')
ax.set_ylabel(r'$\dot{\theta}$ (degrees/s)')
ax.set_title('Phase Portrait')
ax.legend()

# Panel 3: energy vs time
ax = axes[2]
ax.plot(t_sp, T_sp, color='steelblue', linewidth=2, label='Kinetic (T)')
ax.plot(t_sp, V_sp, color='coral', linewidth=2, label='Potential (V)')
ax.plot(t_sp, H_sp, color='seagreen', linewidth=2, linestyle='--', label='Total (H = T + V)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Energy (J)')
ax.set_title('Energy vs Time')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- Compare small-angle (linear) vs full nonlinear dynamics ----
def pendulum_linear_dynamics(t, state, m, l, g):
    """Linearized pendulum dynamics: theta_ddot = -(g/l)*theta.

    Args:
        t: Current time. Scalar.
        state: [theta, theta_dot]. Shape: (2,).
        m: Mass. Scalar.
        l: Length. Scalar.
        g: Gravitational acceleration. Scalar.

    Returns:
        dstate: [theta_dot, theta_ddot]. Shape: (2,).
    """
    theta, theta_dot = state
    theta_ddot = -(g / l) * theta  # sin(theta) ~ theta
    return np.array([theta_dot, theta_ddot])


sp_linear = lambda t, state: pendulum_linear_dynamics(t, state, SP_MASS, SP_LENGTH, GRAVITY)
t_lin, y_lin = rk4_integrate(sp_linear, [theta_0, theta_dot_0], (0, T_SIM), DT)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: angle comparison
ax = axes[0]
ax.plot(t_sp, np.degrees(theta_sp), color='steelblue', linewidth=2, label='Nonlinear')
ax.plot(t_lin, np.degrees(y_lin[:, 0]), color='coral', linewidth=2, linestyle='--', label='Linear (small-angle)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title(r'Nonlinear vs Linear ($\theta_0 = 60°$)')
ax.legend()

# Right: period comparison
ax = axes[1]
# Find zero-crossings for period estimation
crossings_nl = np.where(np.diff(np.sign(theta_sp)))[0]
crossings_lin = np.where(np.diff(np.sign(y_lin[:, 0])))[0]

if len(crossings_nl) >= 4:
    period_nl = 2 * (t_sp[crossings_nl[2]] - t_sp[crossings_nl[0]]) / 2
else:
    period_nl = float('nan')
period_lin = 2 * np.pi * np.sqrt(SP_LENGTH / GRAVITY)

ax.bar(['Nonlinear', 'Linear'], [period_nl, period_lin], color=['steelblue', 'coral'], width=0.5)
ax.set_ylabel('Period (s)')
ax.set_title('Period Comparison')
for i, val in enumerate([period_nl, period_lin]):
    ax.text(i, val + 0.02, f'{val:.4f} s', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print(f"Nonlinear period: {period_nl:.4f} s")
print(f"Linear period:    {period_lin:.4f} s")
print(f"Relative difference: {abs(period_nl - period_lin) / period_lin:.2%}")

---
## 5. Double Pendulum

The double pendulum is a paradigmatic example of **chaotic dynamics** in classical mechanics. Two masses $m_1, m_2$ are connected by rigid rods of lengths $l_1, l_2$. The generalized coordinates are $q = (\theta_1, \theta_2)$, both measured from the downward vertical.

### Kinematics

**Position of mass 1:**
$$x_1 = l_1 \sin\theta_1, \qquad y_1 = -l_1 \cos\theta_1$$

**Position of mass 2:**
$$x_2 = l_1 \sin\theta_1 + l_2 \sin\theta_2, \qquad y_2 = -l_1 \cos\theta_1 - l_2 \cos\theta_2$$

### Energies

**Kinetic energy:**
$$T = \frac{1}{2}(m_1 + m_2) l_1^2 \dot{\theta}_1^2 + \frac{1}{2} m_2 l_2^2 \dot{\theta}_2^2 + m_2 l_1 l_2 \dot{\theta}_1 \dot{\theta}_2 \cos(\theta_1 - \theta_2)$$

**Potential energy:**
$$V = -(m_1 + m_2) g l_1 \cos\theta_1 - m_2 g l_2 \cos\theta_2$$

### Manipulator Equation Form

Applying the Euler-Lagrange equation to each coordinate yields the standard form:

$$\boxed{M(q)\ddot{q} + C(q, \dot{q})\dot{q} + g(q) = 0}$$

where:

$$M(q) = \begin{bmatrix} (m_1+m_2)l_1^2 & m_2 l_1 l_2 \cos(\theta_1-\theta_2) \\ m_2 l_1 l_2 \cos(\theta_1-\theta_2) & m_2 l_2^2 \end{bmatrix}$$

$$C(q, \dot{q})\dot{q} = \begin{bmatrix} m_2 l_1 l_2 \dot{\theta}_2^2 \sin(\theta_1-\theta_2) \\ -m_2 l_1 l_2 \dot{\theta}_1^2 \sin(\theta_1-\theta_2) \end{bmatrix}$$

$$g(q) = \begin{bmatrix} (m_1+m_2) g l_1 \sin\theta_1 \\ m_2 g l_2 \sin\theta_2 \end{bmatrix}$$

In [ ]:
def M_double(q, m1, m2, l1, l2):
    """Mass (inertia) matrix for the double pendulum.

    Args:
        q: Joint angles [theta1, theta2]. Shape: (2,).
        m1: Mass of link 1. Scalar.
        m2: Mass of link 2. Scalar.
        l1: Length of link 1. Scalar.
        l2: Length of link 2. Scalar.

    Returns:
        M: Mass matrix. Shape: (2, 2).
    """
    delta = q[0] - q[1]
    M = np.array([
        [(m1 + m2) * l1**2,          m2 * l1 * l2 * np.cos(delta)],
        [m2 * l1 * l2 * np.cos(delta), m2 * l2**2                ]
    ])
    return M


def C_double(q, qd, m1, m2, l1, l2):
    """Coriolis/centripetal force vector for the double pendulum.

    Returns C(q, qd) * qd as a vector (not the matrix C itself).

    Args:
        q: Joint angles [theta1, theta2]. Shape: (2,).
        qd: Joint velocities [theta1_dot, theta2_dot]. Shape: (2,).
        m1: Mass of link 1. Scalar.
        m2: Mass of link 2. Scalar.
        l1: Length of link 1. Scalar.
        l2: Length of link 2. Scalar.

    Returns:
        c: Coriolis/centripetal force vector. Shape: (2,).
    """
    delta = q[0] - q[1]
    c = np.array([
         m2 * l1 * l2 * qd[1]**2 * np.sin(delta),
        -m2 * l1 * l2 * qd[0]**2 * np.sin(delta)
    ])
    return c


def g_double(q, m1, m2, l1, l2, g):
    """Gravity vector for the double pendulum.

    Args:
        q: Joint angles [theta1, theta2]. Shape: (2,).
        m1: Mass of link 1. Scalar.
        m2: Mass of link 2. Scalar.
        l1: Length of link 1. Scalar.
        l2: Length of link 2. Scalar.
        g: Gravitational acceleration. Scalar.

    Returns:
        gvec: Gravity force vector. Shape: (2,).
    """
    gvec = np.array([
        (m1 + m2) * g * l1 * np.sin(q[0]),
        m2 * g * l2 * np.sin(q[1])
    ])
    return gvec


def double_pendulum_dynamics(t, state, m1, m2, l1, l2, g):
    """Equations of motion for the double pendulum.

    State: [theta1, theta2, theta1_dot, theta2_dot]

    Args:
        t: Current time. Scalar.
        state: [theta1, theta2, theta1_dot, theta2_dot]. Shape: (4,).
        m1: Mass of link 1. Scalar.
        m2: Mass of link 2. Scalar.
        l1: Length of link 1. Scalar.
        l2: Length of link 2. Scalar.
        g: Gravitational acceleration. Scalar.

    Returns:
        dstate: Time derivative of state. Shape: (4,).
    """
    q = state[:2]
    qd = state[2:]

    M = M_double(q, m1, m2, l1, l2)
    c = C_double(q, qd, m1, m2, l1, l2)
    gv = g_double(q, m1, m2, l1, l2, g)

    # M * qdd = -c - gv  =>  qdd = M^{-1} * (-c - gv)
    qdd = np.linalg.solve(M, -c - gv)

    return np.concatenate([qd, qdd])


def double_pendulum_energy(state, m1, m2, l1, l2, g):
    """Compute kinetic and potential energy for the double pendulum.

    Args:
        state: States [theta1, theta2, theta1_dot, theta2_dot]. Shape: (N, 4) or (4,).
        m1: Mass of link 1. Scalar.
        m2: Mass of link 2. Scalar.
        l1: Length of link 1. Scalar.
        l2: Length of link 2. Scalar.
        g: Gravitational acceleration. Scalar.

    Returns:
        T: Kinetic energy. Shape: (N,) or scalar.
        V: Potential energy. Shape: (N,) or scalar.
    """
    if state.ndim == 1:
        state = state.reshape(1, -1)
        squeeze = True
    else:
        squeeze = False

    th1, th2 = state[:, 0], state[:, 1]
    thd1, thd2 = state[:, 2], state[:, 3]
    delta = th1 - th2

    T = (0.5 * (m1 + m2) * l1**2 * thd1**2
         + 0.5 * m2 * l2**2 * thd2**2
         + m2 * l1 * l2 * thd1 * thd2 * np.cos(delta))
    V = -(m1 + m2) * g * l1 * np.cos(th1) - m2 * g * l2 * np.cos(th2)

    if squeeze:
        return T[0], V[0]
    return T, V


print("Double pendulum functions defined.")

In [ ]:
# ---- Simulate double pendulum ----
dp_ic = np.array([np.pi / 2, np.pi / 2, 0.0, 0.0])  # both at 90 degrees, at rest

dp_dynamics = lambda t, s: double_pendulum_dynamics(t, s, DP_M1, DP_M2, DP_L1, DP_L2, GRAVITY)
t_dp, y_dp = rk4_integrate(dp_dynamics, dp_ic, (0, T_SIM), DT)

# ---- Energy conservation ----
T_dp, V_dp = double_pendulum_energy(y_dp, DP_M1, DP_M2, DP_L1, DP_L2, GRAVITY)
H_dp = T_dp + V_dp
H0_dp = H_dp[0]
energy_error_dp = np.max(np.abs((H_dp - H0_dp) / H0_dp))
status_dp = "PASS" if energy_error_dp < ENERGY_TOL else "FAIL"
print(f"Energy conservation: max |DeltaH/H0| = {energy_error_dp:.2e} [{status_dp}]")
print(f"Initial energy: H0 = {H0_dp:.6f} J")

In [ ]:
# ---- 3-Panel: Double pendulum results ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: angles vs time
ax = axes[0]
ax.plot(t_dp, np.degrees(y_dp[:, 0]), color='steelblue', linewidth=1.5, label=r'$\theta_1$')
ax.plot(t_dp, np.degrees(y_dp[:, 1]), color='coral', linewidth=1.5, label=r'$\theta_2$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Double Pendulum: Angles')
ax.legend()

# Panel 2: phase portrait (theta1 vs theta1_dot)
ax = axes[1]
ax.plot(np.degrees(y_dp[:, 0]), np.degrees(y_dp[:, 2]), color='steelblue', linewidth=0.5, alpha=0.7)
ax.plot(np.degrees(y_dp[0, 0]), np.degrees(y_dp[0, 2]), 'ko', markersize=8, label='Start')
ax.set_xlabel(r'$\theta_1$ (degrees)')
ax.set_ylabel(r'$\dot{\theta}_1$ (degrees/s)')
ax.set_title('Phase Portrait (Link 1)')
ax.legend()

# Panel 3: energy
ax = axes[2]
ax.plot(t_dp, T_dp, color='steelblue', linewidth=1.5, label='Kinetic (T)')
ax.plot(t_dp, V_dp, color='coral', linewidth=1.5, label='Potential (V)')
ax.plot(t_dp, H_dp, color='seagreen', linewidth=2, linestyle='--', label='Total (H)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Energy (J)')
ax.set_title('Energy Conservation')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- Chaos visualization: two nearby trajectories ----
epsilon = 1e-4
dp_ic_perturbed = dp_ic.copy()
dp_ic_perturbed[0] += epsilon  # perturb theta1 by epsilon

dp_dynamics_p = lambda t, s: double_pendulum_dynamics(t, s, DP_M1, DP_M2, DP_L1, DP_L2, GRAVITY)
t_dp_p, y_dp_p = rk4_integrate(dp_dynamics_p, dp_ic_perturbed, (0, T_SIM), DT)

# Compute trajectory distance in state space
delta_state = y_dp - y_dp_p
trajectory_distance = np.sqrt(np.sum(delta_state**2, axis=1))

# Tip positions for visualization
x2_orig = DP_L1 * np.sin(y_dp[:, 0]) + DP_L2 * np.sin(y_dp[:, 1])
y2_orig = -DP_L1 * np.cos(y_dp[:, 0]) - DP_L2 * np.cos(y_dp[:, 1])
x2_pert = DP_L1 * np.sin(y_dp_p[:, 0]) + DP_L2 * np.sin(y_dp_p[:, 1])
y2_pert = -DP_L1 * np.cos(y_dp_p[:, 0]) - DP_L2 * np.cos(y_dp_p[:, 1])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: tip traces
ax = axes[0]
ax.plot(x2_orig, y2_orig, color='steelblue', linewidth=0.5, alpha=0.7, label='Original')
ax.plot(x2_pert, y2_pert, color='coral', linewidth=0.5, alpha=0.7, label=f'Perturbed ($\\varepsilon = {epsilon}$)')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Double Pendulum Tip Traces')
ax.set_aspect('equal')
ax.legend(fontsize=10)

# Panel 2: theta1 divergence
ax = axes[1]
ax.plot(t_dp, np.degrees(y_dp[:, 0]), color='steelblue', linewidth=1.5, label='Original')
ax.plot(t_dp_p, np.degrees(y_dp_p[:, 0]), color='coral', linewidth=1.5, linestyle='--', label='Perturbed')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$\theta_1$ (degrees)')
ax.set_title('Sensitivity to Initial Conditions')
ax.legend()

# Panel 3: Lyapunov-like divergence
ax = axes[2]
# Avoid log(0)
log_dist = np.log10(np.maximum(trajectory_distance, 1e-20))
ax.plot(t_dp, log_dist, color='seagreen', linewidth=1.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$\log_{10}$(trajectory distance)')
ax.set_title('Lyapunov-like Divergence')
ax.axhline(np.log10(epsilon), color='gray', linestyle='--', linewidth=1,
           label=f'Initial separation = {epsilon}')
ax.legend()

plt.tight_layout()
plt.show()

# Estimate Lyapunov exponent from early exponential growth
t_fit_end = 3.0  # fit over first 3 seconds
mask_fit = t_dp < t_fit_end
t_fit = t_dp[mask_fit]
dist_fit = trajectory_distance[mask_fit]
# ln(d(t)) ~ ln(d0) + lambda * t
valid = dist_fit > 0
if np.sum(valid) > 10:
    coeffs = np.polyfit(t_fit[valid], np.log(dist_fit[valid]), 1)
    lyap_est = coeffs[0]
    print(f"Estimated Lyapunov exponent: lambda ~ {lyap_est:.2f} /s")
    print(f"Positive Lyapunov exponent confirms chaotic behavior.")

---
## 6. 2R Planar Robot Arm

We now consider an **actuated** 2-link planar robot arm. Unlike the double pendulum (where both joints are free), a robot arm has motors at each joint that apply torques $\tau_1, \tau_2$.

The full **manipulator equation** is:

$$\boxed{M(q)\ddot{q} + C(q, \dot{q})\dot{q} + g(q) = \tau}$$

The key difference from the double pendulum: the right-hand side is now $\tau \neq 0$.

### Parameters

For a robot arm with finite-size links (not point masses), the inertia calculation includes:
- $l_{c1}, l_{c2}$: distances from each joint to the center of mass of each link
- $I_1, I_2$: moments of inertia about the center of mass of each link

The mass matrix becomes:

$$M_{11} = m_1 l_{c1}^2 + I_1 + m_2(l_1^2 + l_{c2}^2 + 2l_1 l_{c2}\cos q_2) + I_2$$
$$M_{12} = M_{21} = m_2(l_{c2}^2 + l_1 l_{c2}\cos q_2) + I_2$$
$$M_{22} = m_2 l_{c2}^2 + I_2$$

In [ ]:
def manipulator_M(q, params):
    """Mass (inertia) matrix for the 2R planar robot arm.

    Args:
        q: Joint angles [q1, q2]. Shape: (2,).
        params: Dictionary with keys 'm1', 'm2', 'l1', 'l2', 'lc1', 'lc2', 'I1', 'I2'.

    Returns:
        M: Mass matrix. Shape: (2, 2).
    """
    m1, m2 = params['m1'], params['m2']
    l1, lc1, lc2 = params['l1'], params['lc1'], params['lc2']
    I1, I2 = params['I1'], params['I2']

    cos_q2 = np.cos(q[1])

    M11 = m1 * lc1**2 + I1 + m2 * (l1**2 + lc2**2 + 2 * l1 * lc2 * cos_q2) + I2
    M12 = m2 * (lc2**2 + l1 * lc2 * cos_q2) + I2
    M22 = m2 * lc2**2 + I2

    return np.array([[M11, M12],
                     [M12, M22]])


def manipulator_C(q, qd, params):
    """Coriolis/centripetal matrix for the 2R planar robot arm.

    Uses the Christoffel symbol formulation.

    Args:
        q: Joint angles [q1, q2]. Shape: (2,).
        qd: Joint velocities [q1_dot, q2_dot]. Shape: (2,).
        params: Dictionary with keys 'm2', 'l1', 'lc2'.

    Returns:
        C: Coriolis matrix. Shape: (2, 2).
    """
    m2 = params['m2']
    l1, lc2 = params['l1'], params['lc2']

    h = m2 * l1 * lc2 * np.sin(q[1])

    C = np.array([[-h * qd[1],    -h * (qd[0] + qd[1])],
                  [ h * qd[0],     0.0                  ]])
    return C


def manipulator_g(q, params):
    """Gravity vector for the 2R planar robot arm.

    Args:
        q: Joint angles [q1, q2]. Shape: (2,).
        params: Dictionary with keys 'm1', 'm2', 'l1', 'lc1', 'lc2', 'g'.

    Returns:
        gvec: Gravity torque vector. Shape: (2,).
    """
    m1, m2 = params['m1'], params['m2']
    l1, lc1, lc2 = params['l1'], params['lc1'], params['lc2']
    g = params['g']

    g1 = (m1 * lc1 + m2 * l1) * g * np.sin(q[0]) + m2 * lc2 * g * np.sin(q[0] + q[1])
    g2 = m2 * lc2 * g * np.sin(q[0] + q[1])

    return np.array([g1, g2])


def arm_dynamics(t, state, params, tau_func=None):
    """Equations of motion for the 2R planar robot arm.

    State: [q1, q2, q1_dot, q2_dot]

    Args:
        t: Current time. Scalar.
        state: [q1, q2, q1_dot, q2_dot]. Shape: (4,).
        params: Robot parameters dictionary.
        tau_func: Torque function tau(t, state) -> Shape: (2,). If None, tau = 0.

    Returns:
        dstate: Time derivative of state. Shape: (4,).
    """
    q = state[:2]
    qd = state[2:]

    M = manipulator_M(q, params)
    C = manipulator_C(q, qd, params)
    gv = manipulator_g(q, params)

    if tau_func is not None:
        tau = tau_func(t, state)
    else:
        tau = np.zeros(2)

    # M * qdd + C * qd + g = tau  =>  qdd = M^{-1} * (tau - C*qd - g)
    qdd = np.linalg.solve(M, tau - C @ qd - gv)

    return np.concatenate([qd, qdd])


# Arm parameters dictionary
ARM_PARAMS = {
    'm1': ARM_M1, 'm2': ARM_M2,
    'l1': ARM_L1, 'l2': ARM_L2,
    'lc1': ARM_LC1, 'lc2': ARM_LC2,
    'I1': ARM_I1, 'I2': ARM_I2,
    'g': GRAVITY
}

print("2R arm functions defined.")
print(f"Arm parameters: m1={ARM_M1}, m2={ARM_M2}, l1={ARM_L1}, l2={ARM_L2}")
print(f"                lc1={ARM_LC1}, lc2={ARM_LC2}, I1={ARM_I1:.4f}, I2={ARM_I2:.4f}")

In [ ]:
# ---- Verify M matrix properties ----
test_configs = [
    np.array([0.0, 0.0]),
    np.array([np.pi/4, np.pi/3]),
    np.array([np.pi/2, -np.pi/4]),
    np.array([np.pi, np.pi/2]),
    np.array([-np.pi/3, 2*np.pi/3]),
]

print("=== Mass Matrix Properties ===")
all_symmetric = True
all_positive_definite = True

for i, q_test in enumerate(test_configs):
    M_test = manipulator_M(q_test, ARM_PARAMS)

    # Symmetry check
    sym_err = np.max(np.abs(M_test - M_test.T))
    is_sym = sym_err < 1e-12
    all_symmetric = all_symmetric and is_sym

    # Positive definite check
    eigvals = np.linalg.eigvalsh(M_test)
    is_pd = np.all(eigvals > 0)
    all_positive_definite = all_positive_definite and is_pd

    print(f"  q = [{q_test[0]:+.2f}, {q_test[1]:+.2f}]: "
          f"sym_err = {sym_err:.1e}, "
          f"eigenvalues = [{eigvals[0]:.4f}, {eigvals[1]:.4f}]")

status_sym = "PASS" if all_symmetric else "FAIL"
status_pd = "PASS" if all_positive_definite else "FAIL"
print(f"\nSymmetric:         [{status_sym}]")
print(f"Positive definite: [{status_pd}]")

In [ ]:
# ---- Verify skew-symmetry of (Mdot - 2C) ----
print("=== Skew-Symmetry of (Mdot - 2C) ===")
all_skew = True
dt_fd = 1e-7

for i, q_test in enumerate(test_configs):
    qd_test = np.array([1.5, -0.8])  # arbitrary velocities

    # Numerical Mdot via finite differences
    # M depends on q, so Mdot = dM/dt = sum_j (dM/dq_j) * qd_j
    q_plus = q_test + dt_fd * qd_test
    q_minus = q_test - dt_fd * qd_test
    M_plus = manipulator_M(q_plus, ARM_PARAMS)
    M_minus = manipulator_M(q_minus, ARM_PARAMS)
    M_dot = (M_plus - M_minus) / (2 * dt_fd)

    C_test = manipulator_C(q_test, qd_test, ARM_PARAMS)

    # Check: N = Mdot - 2C should be skew-symmetric (N + N^T = 0)
    N = M_dot - 2 * C_test
    skew_err = np.max(np.abs(N + N.T))
    is_skew = skew_err < 1e-4
    all_skew = all_skew and is_skew

    print(f"  q = [{q_test[0]:+.2f}, {q_test[1]:+.2f}]: "
          f"max |N + N^T| = {skew_err:.2e}")

status_skew = "PASS" if all_skew else "FAIL"
print(f"\nSkew-symmetry of (Mdot - 2C): [{status_skew}]")
print("(This property guarantees energy conservation in unforced motion.)")

In [ ]:
# ---- Forward dynamics: apply constant torque ----
tau_const = np.array([2.0, 0.5])  # N*m

arm_ic = np.array([0.0, 0.0, 0.0, 0.0])  # start at rest, hanging down
arm_fwd = lambda t, s: arm_dynamics(t, s, ARM_PARAMS, lambda t, s: tau_const)
t_arm, y_arm = rk4_integrate(arm_fwd, arm_ic, (0, 3.0), DT)

# Compute end-effector position
x_ee = ARM_L1 * np.cos(y_arm[:, 0]) + ARM_L2 * np.cos(y_arm[:, 0] + y_arm[:, 1])
y_ee = ARM_L1 * np.sin(y_arm[:, 0]) + ARM_L2 * np.sin(y_arm[:, 0] + y_arm[:, 1])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: joint angles
ax = axes[0]
ax.plot(t_arm, np.degrees(y_arm[:, 0]), color='steelblue', linewidth=2, label=r'$q_1$')
ax.plot(t_arm, np.degrees(y_arm[:, 1]), color='coral', linewidth=2, label=r'$q_2$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title(f'Forward Dynamics ($\\tau$ = [{tau_const[0]}, {tau_const[1]}] N$\\cdot$m)')
ax.legend()

# Panel 2: end-effector path
ax = axes[1]
ax.plot(x_ee, y_ee, color='seagreen', linewidth=1.5)
ax.plot(x_ee[0], y_ee[0], 'ko', markersize=8, label='Start')
ax.plot(x_ee[-1], y_ee[-1], 'rs', markersize=8, label='End')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('End-Effector Path')
ax.set_aspect('equal')
ax.legend()

# Panel 3: joint velocities
ax = axes[2]
ax.plot(t_arm, np.degrees(y_arm[:, 2]), color='steelblue', linewidth=2, label=r'$\dot{q}_1$')
ax.plot(t_arm, np.degrees(y_arm[:, 3]), color='coral', linewidth=2, label=r'$\dot{q}_2$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angular velocity (degrees/s)')
ax.set_title('Joint Velocities')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- Inverse dynamics: given desired q(t), compute required tau(t) ----
def desired_trajectory(t):
    """Compute desired joint trajectory and its derivatives.

    q1_d(t) = pi/4 * sin(t)
    q2_d(t) = pi/6 * sin(2*t)

    Args:
        t: Time. Scalar or Shape: (N,).

    Returns:
        q_d: Desired position. Shape: (2,) or (N, 2).
        qd_d: Desired velocity. Shape: (2,) or (N, 2).
        qdd_d: Desired acceleration. Shape: (2,) or (N, 2).
    """
    scalar = np.isscalar(t)
    t = np.atleast_1d(t)

    q_d = np.column_stack([np.pi/4 * np.sin(t), np.pi/6 * np.sin(2*t)])
    qd_d = np.column_stack([np.pi/4 * np.cos(t), np.pi/3 * np.cos(2*t)])
    qdd_d = np.column_stack([-np.pi/4 * np.sin(t), -2*np.pi/3 * np.sin(2*t)])

    if scalar:
        return q_d[0], qd_d[0], qdd_d[0]
    return q_d, qd_d, qdd_d


def inverse_dynamics(q, qd, qdd, params):
    """Compute the required torque for given motion.

    tau = M(q)*qdd + C(q,qd)*qd + g(q)

    Args:
        q: Joint angles. Shape: (2,).
        qd: Joint velocities. Shape: (2,).
        qdd: Joint accelerations. Shape: (2,).
        params: Robot parameters dictionary.

    Returns:
        tau: Required torque. Shape: (2,).
    """
    M = manipulator_M(q, params)
    C = manipulator_C(q, qd, params)
    gv = manipulator_g(q, params)
    return M @ qdd + C @ qd + gv


# Compute inverse dynamics along desired trajectory
t_inv = np.linspace(0, 2 * np.pi, 1000)
q_d, qd_d, qdd_d = desired_trajectory(t_inv)

tau_inv = np.zeros((len(t_inv), 2))
for i in range(len(t_inv)):
    tau_inv[i] = inverse_dynamics(q_d[i], qd_d[i], qdd_d[i], ARM_PARAMS)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: desired trajectory
ax = axes[0]
ax.plot(t_inv, np.degrees(q_d[:, 0]), color='steelblue', linewidth=2, label=r'$q_1^d$')
ax.plot(t_inv, np.degrees(q_d[:, 1]), color='coral', linewidth=2, label=r'$q_2^d$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Angle (degrees)')
ax.set_title('Desired Trajectory')
ax.legend()

# Right: required torques
ax = axes[1]
ax.plot(t_inv, tau_inv[:, 0], color='steelblue', linewidth=2, label=r'$\tau_1$')
ax.plot(t_inv, tau_inv[:, 1], color='coral', linewidth=2, label=r'$\tau_2$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Torque (N$\\cdot$m)')
ax.set_title('Inverse Dynamics: Required Torques')
ax.legend()

plt.tight_layout()
plt.show()

---
## 7. Computed Torque Control

The idea of **computed torque control** (also called inverse dynamics control or feedback linearization) is to use the robot's dynamic model to cancel out all the nonlinear terms, leaving simple linear error dynamics.

### Control Law

$$\boxed{\tau = M(q)\left(\ddot{q}_d + K_d(\dot{q}_d - \dot{q}) + K_p(q_d - q)\right) + C(q, \dot{q})\dot{q} + g(q)}$$

Substituting into $M\ddot{q} + C\dot{q} + g = \tau$, the nonlinear terms cancel, leaving:

$$\ddot{q} = \ddot{q}_d + K_d(\dot{q}_d - \dot{q}) + K_p(q_d - q)$$

Defining the tracking error $e = q_d - q$:

$$\ddot{e} + K_d \dot{e} + K_p e = 0$$

This is a **linear, decoupled** second-order system. Choosing $K_p$ and $K_d$ for critical damping:

$$K_p = \omega_n^2 I, \qquad K_d = 2\omega_n I$$

where $\omega_n$ is the desired natural frequency.

### Comparison with Simple PD Control

A simple PD controller applies:

$$\tau_{\text{PD}} = K_p(q_d - q) + K_d(\dot{q}_d - \dot{q})$$

This does **not** cancel the nonlinear dynamics, so tracking performance degrades as the arm moves faster or further from equilibrium.

In [ ]:
def computed_torque_controller(t, state, params, desired_traj_func, Kp, Kd):
    """Computed torque (inverse dynamics) controller.

    tau = M(q)*(qdd_d + Kd*(qd_d - qd) + Kp*(q_d - q)) + C(q,qd)*qd + g(q)

    Args:
        t: Current time. Scalar.
        state: [q1, q2, q1_dot, q2_dot]. Shape: (4,).
        params: Robot parameters dictionary.
        desired_traj_func: Function(t) -> (q_d, qd_d, qdd_d).
        Kp: Position gain matrix. Shape: (2, 2).
        Kd: Velocity gain matrix. Shape: (2, 2).

    Returns:
        tau: Control torque. Shape: (2,).
    """
    q = state[:2]
    qd = state[2:]

    q_d, qd_d, qdd_d = desired_traj_func(t)

    e = q_d - q
    ed = qd_d - qd

    # Desired acceleration with feedback
    a = qdd_d + Kd @ ed + Kp @ e

    M = manipulator_M(q, params)
    C = manipulator_C(q, qd, params)
    gv = manipulator_g(q, params)

    tau = M @ a + C @ qd + gv
    return tau


def pd_controller(t, state, params, desired_traj_func, Kp, Kd):
    """Simple PD controller (no model-based cancellation).

    tau = Kp*(q_d - q) + Kd*(qd_d - qd)

    Args:
        t: Current time. Scalar.
        state: [q1, q2, q1_dot, q2_dot]. Shape: (4,).
        params: Robot parameters dictionary.
        desired_traj_func: Function(t) -> (q_d, qd_d, qdd_d).
        Kp: Position gain matrix. Shape: (2, 2).
        Kd: Velocity gain matrix. Shape: (2, 2).

    Returns:
        tau: Control torque. Shape: (2,).
    """
    q = state[:2]
    qd = state[2:]

    q_d, qd_d, qdd_d = desired_traj_func(t)

    e = q_d - q
    ed = qd_d - qd

    tau = Kp @ e + Kd @ ed
    return tau


# ---- Controller gains (critically damped) ----
omega_n = 20.0  # natural frequency (rad/s)
KP = omega_n**2 * np.eye(2)
KD = 2 * omega_n * np.eye(2)

print(f"Controller gains: Kp = {omega_n**2:.0f} * I, Kd = {2*omega_n:.0f} * I")
print(f"Natural frequency: omega_n = {omega_n} rad/s")

In [ ]:
# ---- Simulate computed torque control vs PD control ----
T_CTRL = 2 * np.pi  # one full period of the trajectory

# Initial condition: start at q_d(0)
q0_d, qd0_d, _ = desired_trajectory(0.0)
ctrl_ic = np.concatenate([q0_d, qd0_d])

# Computed torque controller
ct_tau = lambda t, s: computed_torque_controller(t, s, ARM_PARAMS, desired_trajectory, KP, KD)
ct_dyn = lambda t, s: arm_dynamics(t, s, ARM_PARAMS, ct_tau)
t_ct, y_ct = rk4_integrate(ct_dyn, ctrl_ic, (0, T_CTRL), DT)

# PD controller
pd_tau = lambda t, s: pd_controller(t, s, ARM_PARAMS, desired_trajectory, KP, KD)
pd_dyn = lambda t, s: arm_dynamics(t, s, ARM_PARAMS, pd_tau)
t_pd, y_pd = rk4_integrate(pd_dyn, ctrl_ic, (0, T_CTRL), DT)

# Desired trajectory for plotting
q_d_plot, _, _ = desired_trajectory(t_ct)

# Tracking errors
error_ct = y_ct[:, :2] - q_d_plot
error_pd = y_pd[:, :2] - q_d_plot

max_error_ct = np.max(np.abs(error_ct))
max_error_pd = np.max(np.abs(error_pd))

print(f"Max tracking error (computed torque): {np.degrees(max_error_ct):.4f} degrees")
print(f"Max tracking error (PD control):     {np.degrees(max_error_pd):.4f} degrees")
print(f"Improvement ratio: {max_error_pd / max_error_ct:.1f}x")

In [ ]:
# ---- 3-Panel: Control comparison ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: joint 1 tracking
ax = axes[0]
ax.plot(t_ct, np.degrees(q_d_plot[:, 0]), 'k--', linewidth=1.5, label='Desired')
ax.plot(t_ct, np.degrees(y_ct[:, 0]), color='steelblue', linewidth=2, label='Computed Torque')
ax.plot(t_pd, np.degrees(y_pd[:, 0]), color='coral', linewidth=2, label='PD')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$q_1$ (degrees)')
ax.set_title('Joint 1 Tracking')
ax.legend()

# Panel 2: joint 2 tracking
ax = axes[1]
ax.plot(t_ct, np.degrees(q_d_plot[:, 1]), 'k--', linewidth=1.5, label='Desired')
ax.plot(t_ct, np.degrees(y_ct[:, 1]), color='steelblue', linewidth=2, label='Computed Torque')
ax.plot(t_pd, np.degrees(y_pd[:, 1]), color='coral', linewidth=2, label='PD')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$q_2$ (degrees)')
ax.set_title('Joint 2 Tracking')
ax.legend()

# Panel 3: tracking error norm
ax = axes[2]
err_norm_ct = np.degrees(np.sqrt(error_ct[:, 0]**2 + error_ct[:, 1]**2))
err_norm_pd = np.degrees(np.sqrt(error_pd[:, 0]**2 + error_pd[:, 1]**2))
ax.plot(t_ct, err_norm_ct, color='steelblue', linewidth=2, label='Computed Torque')
ax.plot(t_pd, err_norm_pd, color='coral', linewidth=2, label='PD')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Tracking Error (degrees)')
ax.set_title('Tracking Error Comparison')
ax.legend()
ax.set_yscale('log')

plt.tight_layout()
plt.show()

---
## 8. Energy Analysis

Energy methods provide powerful verification tools and physical insight.

### Conservation for Unforced Systems

For $\tau = 0$, the Hamiltonian $H = T + V$ is conserved. This follows from the skew-symmetry of $(\dot{M} - 2C)$:

$$\frac{dH}{dt} = \dot{q}^T M \ddot{q} + \frac{1}{2}\dot{q}^T \dot{M} \dot{q} + \dot{q}^T g = \dot{q}^T (M\ddot{q} + C\dot{q} + g) + \frac{1}{2}\dot{q}^T(\dot{M} - 2C)\dot{q} = \dot{q}^T \tau + 0 = 0$$

### Power Balance for Forced Systems

When $\tau \neq 0$:

$$\boxed{\frac{dH}{dt} = \tau \cdot \dot{q}}$$

The rate of energy change equals the power input by the actuators.

In [ ]:
def arm_energy(state, params):
    """Compute kinetic and potential energy for the 2R arm.

    Args:
        state: [q1, q2, q1_dot, q2_dot]. Shape: (N, 4) or (4,).
        params: Robot parameters dictionary.

    Returns:
        T: Kinetic energy. Shape: (N,) or scalar.
        V: Potential energy. Shape: (N,) or scalar.
    """
    if state.ndim == 1:
        state = state.reshape(1, -1)
        squeeze = True
    else:
        squeeze = False

    T_arr = np.zeros(len(state))
    V_arr = np.zeros(len(state))

    for i in range(len(state)):
        q = state[i, :2]
        qd = state[i, 2:]
        M = manipulator_M(q, params)
        T_arr[i] = 0.5 * qd @ M @ qd
        # V = potential energy
        m1, m2 = params['m1'], params['m2']
        l1, lc1, lc2 = params['l1'], params['lc1'], params['lc2']
        g = params['g']
        V_arr[i] = (-(m1 * lc1 + m2 * l1) * g * np.cos(q[0])
                    - m2 * lc2 * g * np.cos(q[0] + q[1]))

    if squeeze:
        return T_arr[0], V_arr[0]
    return T_arr, V_arr


# ---- Unforced arm: energy conservation ----
arm_unforced_ic = np.array([np.pi/4, np.pi/6, 0.5, -0.3])
arm_unf_dyn = lambda t, s: arm_dynamics(t, s, ARM_PARAMS, None)
t_unf, y_unf = rk4_integrate(arm_unf_dyn, arm_unforced_ic, (0, T_SIM), DT)

T_unf, V_unf = arm_energy(y_unf, ARM_PARAMS)
H_unf = T_unf + V_unf
H0_unf = H_unf[0]
energy_err_unf = np.max(np.abs((H_unf - H0_unf) / H0_unf))
status_unf = "PASS" if energy_err_unf < ENERGY_TOL else "FAIL"
print(f"Unforced arm energy conservation: max |DeltaH/H0| = {energy_err_unf:.2e} [{status_unf}]")

In [ ]:
# ---- Forced arm: verify dH/dt = tau . qdot ----
T_ct, V_ct = arm_energy(y_ct, ARM_PARAMS)
H_ct = T_ct + V_ct

# Numerical dH/dt
dH_dt_num = np.gradient(H_ct, t_ct)

# Analytical: tau . qdot
power_input = np.zeros(len(t_ct))
for i in range(len(t_ct)):
    tau_i = computed_torque_controller(t_ct[i], y_ct[i], ARM_PARAMS, desired_trajectory, KP, KD)
    power_input[i] = tau_i @ y_ct[i, 2:]

# Compare (skip endpoints where gradient is less accurate)
mid = slice(100, -100)
power_err = np.max(np.abs(dH_dt_num[mid] - power_input[mid]))
power_scale = np.max(np.abs(power_input[mid]))
rel_power_err = power_err / power_scale if power_scale > 1e-12 else power_err
status_power = "PASS" if rel_power_err < 0.01 else "FAIL"
print(f"Power balance: max |dH/dt - tau.qdot| / max|tau.qdot| = {rel_power_err:.2e} [{status_power}]")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: energy for unforced system
ax = axes[0]
ax.plot(t_unf, T_unf, color='steelblue', linewidth=2, label='Kinetic (T)')
ax.plot(t_unf, V_unf, color='coral', linewidth=2, label='Potential (V)')
ax.plot(t_unf, H_unf, color='seagreen', linewidth=2, linestyle='--', label='Total (H)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Energy (J)')
ax.set_title('Unforced Arm: Energy Conservation')
ax.legend()

# Right: power balance for forced system
ax = axes[1]
ax.plot(t_ct[mid], dH_dt_num[mid], color='steelblue', linewidth=2, label=r'$dH/dt$ (numerical)')
ax.plot(t_ct[mid], power_input[mid], color='coral', linewidth=2, linestyle='--', label=r'$\tau \cdot \dot{q}$ (analytical)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Power (W)')
ax.set_title('Forced Arm: Power Balance')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- Energy landscape: V(theta1, theta2) for double pendulum ----
theta1_range = np.linspace(-np.pi, np.pi, 100)
theta2_range = np.linspace(-np.pi, np.pi, 100)
TH1, TH2 = np.meshgrid(theta1_range, theta2_range)

V_landscape = (-(DP_M1 + DP_M2) * GRAVITY * DP_L1 * np.cos(TH1)
               - DP_M2 * GRAVITY * DP_L2 * np.cos(TH2))

fig = plt.figure(figsize=(14, 5))

# Left: 3D surface
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(np.degrees(TH1), np.degrees(TH2), V_landscape,
                        cmap='viridis', alpha=0.8, edgecolor='none')
ax1.set_xlabel(r'$\theta_1$ (deg)')
ax1.set_ylabel(r'$\theta_2$ (deg)')
ax1.set_zlabel('V (J)')
ax1.set_title('Potential Energy Landscape')
ax1.view_init(elev=30, azim=45)

# Right: contour plot with equilibria
ax2 = fig.add_subplot(122)
contour = ax2.contourf(np.degrees(TH1), np.degrees(TH2), V_landscape, levels=30, cmap='viridis')
plt.colorbar(contour, ax=ax2, label='V (J)')

# Mark equilibria: critical points of V
# dV/dtheta1 = (m1+m2)*g*l1*sin(theta1) = 0 => theta1 = 0, pi
# dV/dtheta2 = m2*g*l2*sin(theta2) = 0 => theta2 = 0, pi
equilibria = [(0, 0), (0, 180), (180, 0), (180, 180),
              (0, -180), (-180, 0), (-180, -180), (-180, 180), (180, -180)]
for eq in equilibria:
    ax2.plot(eq[0], eq[1], 'r*', markersize=12)

ax2.set_xlabel(r'$\theta_1$ (deg)')
ax2.set_ylabel(r'$\theta_2$ (deg)')
ax2.set_title('Potential Energy Contours (stars = equilibria)')

plt.tight_layout()
plt.show()

# Classify equilibria
print("Equilibrium classification:")
print(f"  (0, 0):     V = {-(DP_M1+DP_M2)*GRAVITY*DP_L1 - DP_M2*GRAVITY*DP_L2:.2f} J  [stable minimum - both hanging]")
print(f"  (0, pi):    V = {-(DP_M1+DP_M2)*GRAVITY*DP_L1 + DP_M2*GRAVITY*DP_L2:.2f} J  [saddle point]")
print(f"  (pi, 0):    V = {(DP_M1+DP_M2)*GRAVITY*DP_L1 - DP_M2*GRAVITY*DP_L2:.2f} J  [saddle point]")
print(f"  (pi, pi):   V = {(DP_M1+DP_M2)*GRAVITY*DP_L1 + DP_M2*GRAVITY*DP_L2:.2f} J  [unstable maximum - both up]")

---
## 9. Summary Visualizations

In [ ]:
# ---- 4-Panel summary figure ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Single pendulum phase portrait with energy contours
ax = axes[0, 0]
theta_grid = np.linspace(-np.pi, np.pi, 200)
thetad_grid = np.linspace(-10, 10, 200)
TH, THD = np.meshgrid(theta_grid, thetad_grid)
H_grid = 0.5 * SP_MASS * SP_LENGTH**2 * THD**2 - SP_MASS * GRAVITY * SP_LENGTH * np.cos(TH)
ax.contour(np.degrees(TH), THD, H_grid, levels=20, cmap='viridis', alpha=0.5)
ax.plot(np.degrees(theta_sp), theta_dot_sp, color='coral', linewidth=2, label='Trajectory')
ax.plot(np.degrees(theta_sp[0]), theta_dot_sp[0], 'ko', markersize=8)
ax.set_xlabel(r'$\theta$ (degrees)')
ax.set_ylabel(r'$\dot{\theta}$ (rad/s)')
ax.set_title('Single Pendulum: Phase Portrait')
ax.legend()

# Panel 2: Double pendulum chaos
ax = axes[0, 1]
ax.plot(x2_orig[:5000], y2_orig[:5000], color='steelblue', linewidth=0.5, alpha=0.7, label='Original')
ax.plot(x2_pert[:5000], y2_pert[:5000], color='coral', linewidth=0.5, alpha=0.7, label='Perturbed')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Double Pendulum: Chaotic Divergence')
ax.set_aspect('equal')
ax.legend(fontsize=10)

# Panel 3: 2R arm trajectory tracking comparison
ax = axes[1, 0]
ax.plot(t_ct, err_norm_ct, color='steelblue', linewidth=2, label='Computed Torque')
ax.plot(t_pd, err_norm_pd, color='coral', linewidth=2, label='PD Control')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Tracking Error (degrees)')
ax.set_title('2R Arm: Tracking Error')
ax.legend()
ax.set_yscale('log')

# Panel 4: Energy conservation for all systems
ax = axes[1, 1]
# Normalize energy drift
ax.semilogy(t_sp, np.abs((H_sp - H0_sp) / H0_sp) + 1e-20,
            color='steelblue', linewidth=1.5, label='Single Pendulum')
ax.semilogy(t_dp, np.abs((H_dp - H0_dp) / H0_dp) + 1e-20,
            color='coral', linewidth=1.5, label='Double Pendulum')
ax.semilogy(t_unf, np.abs((H_unf - H0_unf) / H0_unf) + 1e-20,
            color='seagreen', linewidth=1.5, label='2R Arm (unforced)')
ax.axhline(ENERGY_TOL, color='gray', linestyle='--', linewidth=1, label=f'Tolerance = {ENERGY_TOL}')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$|\Delta H / H_0|$')
ax.set_title('Energy Conservation (all systems)')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Arm animation frames: draw arm at several time steps ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: arm configurations along computed torque trajectory
ax = axes[0]
n_frames = 12
frame_indices = np.linspace(0, len(t_ct) - 1, n_frames, dtype=int)
colors_frames = plt.cm.viridis(np.linspace(0, 1, n_frames))

for idx, fi in enumerate(frame_indices):
    q1_i = y_ct[fi, 0]
    q2_i = y_ct[fi, 1]
    # Joint positions
    x0, y0 = 0.0, 0.0
    x1 = ARM_L1 * np.cos(q1_i)
    y1 = ARM_L1 * np.sin(q1_i)
    x2 = x1 + ARM_L2 * np.cos(q1_i + q2_i)
    y2 = y1 + ARM_L2 * np.sin(q1_i + q2_i)

    alpha = 0.3 + 0.7 * idx / n_frames
    ax.plot([x0, x1, x2], [y0, y1, y2], 'o-', color=colors_frames[idx],
            linewidth=2, markersize=5, alpha=alpha)

# Draw workspace circle
theta_ws = np.linspace(0, 2 * np.pi, 100)
ax.plot((ARM_L1 + ARM_L2) * np.cos(theta_ws), (ARM_L1 + ARM_L2) * np.sin(theta_ws),
        'k--', linewidth=0.5, alpha=0.3)
ax.plot(0, 0, 'ks', markersize=10, label='Base')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Arm Configurations (Computed Torque)')
ax.set_aspect('equal')
ax.legend()

# Right: torque profiles
ax = axes[1]
tau_ct_hist = np.zeros((len(t_ct), 2))
for i in range(len(t_ct)):
    tau_ct_hist[i] = computed_torque_controller(
        t_ct[i], y_ct[i], ARM_PARAMS, desired_trajectory, KP, KD)

ax.plot(t_ct, tau_ct_hist[:, 0], color='steelblue', linewidth=2, label=r'$\tau_1$')
ax.plot(t_ct, tau_ct_hist[:, 1], color='coral', linewidth=2, label=r'$\tau_2$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Torque (N$\\cdot$m)')
ax.set_title('Computed Torque Control: Torque Profiles')
ax.legend()

plt.tight_layout()
plt.show()

---
## 10. Extensions

This notebook has covered the core of Lagrangian dynamics for robotic manipulators. Several important extensions connect to other areas of robotics and mechanics.

### Constrained Dynamics and Lagrange Multipliers

When a robot makes contact with the environment (grasping, walking), we must enforce **holonomic constraints** $\phi(q) = 0$. The constrained equations of motion become:

$$M(q)\ddot{q} + C(q, \dot{q})\dot{q} + g(q) = \tau + J_c^T(q) \lambda$$

where $J_c = \partial \phi / \partial q$ is the constraint Jacobian and $\lambda$ are the Lagrange multipliers (contact forces).

### Hamilton's Equations

The **Legendre transform** converts the Lagrangian to the Hamiltonian formulation. Defining the generalized momentum $p = \partial L / \partial \dot{q} = M(q)\dot{q}$:

$$H(q, p) = p^T \dot{q} - L = \frac{1}{2} p^T M^{-1}(q) p + V(q)$$

Hamilton's equations are first-order:

$$\dot{q} = \frac{\partial H}{\partial p}, \qquad \dot{p} = -\frac{\partial H}{\partial q} + \tau$$

This phase-space formulation is the basis of symplectic integrators, which preserve energy exactly.

### N-Link Arms: Recursive Newton-Euler

For arms with many links ($n > 3$), explicitly computing $M(q)$, $C(q, \dot{q})$, and $g(q)$ becomes expensive ($O(n^3)$). The **recursive Newton-Euler algorithm** computes the inverse dynamics $\tau = f(q, \dot{q}, \ddot{q})$ in $O(n)$ by propagating velocities and accelerations outward from the base, then forces and torques inward from the tip.

### Connection to CHOMP

The smoothness functional in the CHOMP trajectory optimizer (see our [CHOMP notebook](../chomp-algo/chomp_3r_planar.ipynb)):

$$F_{\text{smooth}}[\xi] = \frac{1}{2} \int_0^1 \|\ddot{\xi}(t)\|^2 \, dt$$

penalizes acceleration. In the Lagrangian framework, the **dynamic** cost of acceleration involves the mass matrix:

$$\text{Dynamic effort} = \frac{1}{2} \int \ddot{q}^T M(q) \ddot{q} \, dt$$

Incorporating $M(q)$ into the CHOMP metric leads to **dynamically-aware** trajectory optimization.

### Connection to Calculus of Variations

The Lagrangian formulation is a direct application of the Euler-Lagrange equation from the [Calculus of Variations notebook](../../maths/calculus-of-variations/). The action functional $S[q] = \int L(q, \dot{q}) \, dt$ is the central object — the physical trajectory is the one that makes $S$ stationary. Everything in this notebook follows from that single variational principle.

### Key Takeaways

1. The **Lagrangian** $L = T - V$ encodes all dynamics through scalar energies
2. The **Euler-Lagrange equation** systematically produces the equations of motion
3. The **manipulator equation** $M\ddot{q} + C\dot{q} + g = \tau$ has deep structure: $M$ is symmetric positive definite, $\dot{M} - 2C$ is skew-symmetric
4. **Energy conservation** provides a powerful verification tool for simulations
5. **Computed torque control** exploits the dynamic model to achieve exact linearization and superior tracking